# Part 3 V2 Advanced

GitHub Actions / Google Colab compatible wrapper for the original `Part_3.ipynb`.

**Important:** Keep the original `Part_3.ipynb` in the repository root. This wrapper patches it in memory and does not modify the original file.

Reliability improvements include workspace-root discovery, reuse of pre-downloaded assets, corrected webcam dimensions, mono/stereo-safe audio conversion, webcam end-of-file handling, safe AI text fallback, unique temporary filenames, MP4 fast-start, and FFprobe output validation.


In [ ]:
# ==============================================================
# PART 3 V2 ADVANCED — GitHub Actions / Colab compatible
# Keeps Part_3.ipynb unchanged and patches it safely in memory.
# ==============================================================
import hashlib
import json
import os
from pathlib import Path

def _resolve_workspace_root():
    candidates = []
    env_root = os.environ.get("GITHUB_WORKSPACE", "").strip()
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent])

    checked = []
    for candidate in candidates:
        try:
            root = candidate.expanduser().resolve()
            if root in checked:
                continue
            checked.append(root)
            if (root / "Part_3.ipynb").is_file():
                return root
        except (OSError, RuntimeError):
            continue

    locations = "\n".join(f" - {path}" for path in checked) or " - no readable candidate paths"
    raise FileNotFoundError(
        "Could not locate Part_3.ipynb. Keep the original notebook in the repository root "
        "beside Part_3_v2.ipynb. Checked:\n" + locations
    )

WORKSPACE_ROOT = _resolve_workspace_root()
os.chdir(WORKSPACE_ROOT)
ORIGINAL_NOTEBOOK = WORKSPACE_ROOT / "Part_3.ipynb"

print("Part 3 V2 Advanced")
print("Workspace root:", WORKSPACE_ROOT)
print("Original notebook:", ORIGINAL_NOTEBOOK)

try:
    notebook_data = json.loads(ORIGINAL_NOTEBOOK.read_text(encoding="utf-8"))
except Exception as exc:
    raise RuntimeError(f"Could not read valid JSON from {ORIGINAL_NOTEBOOK}: {exc}") from exc

code_cells = [
    cell for cell in notebook_data.get("cells", [])
    if cell.get("cell_type") == "code"
]
if len(code_cells) != 1:
    raise RuntimeError(
        f"Expected exactly one code cell in Part_3.ipynb; found {len(code_cells)}. "
        "The original notebook structure changed, so no unsafe patch was attempted."
    )

cell_source = code_cells[0].get("source", "")
if isinstance(cell_source, list):
    cell_source = "".join(cell_source)
if not isinstance(cell_source, str) or not cell_source.strip():
    raise RuntimeError("Part_3.ipynb contains an empty or invalid code cell.")

def _replace_once(source, old, new, label):
    count = source.count(old)
    if count != 1:
        raise RuntimeError(
            f"Safe patch stopped: expected one match for '{label}', found {count}. "
            "The original Part_3.ipynb may have changed."
        )
    return source.replace(old, new, 1)

def _replace_all_expected(source, old, new, label, expected_count):
    count = source.count(old)
    if count != expected_count:
        raise RuntimeError(
            f"Safe patch stopped: expected {expected_count} matches for '{label}', found {count}. "
            "The original Part_3.ipynb may have changed."
        )
    return source.replace(old, new)

patched_source = cell_source

# 1) Reuse the assets already downloaded by the master workflow.
old_asset_block = """if IS_GITHUB_ACTIONS:
    assets_url = os.environ.get("DRIVE_ASSETS_URL", "").strip()
    if not assets_url:
        raise RuntimeError("Missing GitHub Actions secret DRIVE_ASSETS_URL (public Google Drive assets folder URL).")
    print("Downloading Part 3 assets from Google Drive into:", ASSETS_DIR)
    assets_download = gdown.download_folder(
        url=assets_url,
        output=str(ASSETS_DIR),
        quiet=False,
        use_cookies=False,
    )
    if not assets_download and not any(p.is_file() for p in ASSETS_DIR.rglob("*")):
        raise RuntimeError("Assets download failed. Check DRIVE_ASSETS_URL and set the folder sharing to Anyone with the link — Viewer.")
    print("Part 2 clips will be read from the downloaded GitHub artifact; no second clip download is performed.")
else:
    ASSETS_FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"
    CLIPS_FOLDER_ID = "10DSjb9etdFzQTg_rftRJpJkC5jKBjsNT"
    print("Downloading original Colab assets folder...")
    gdown.download_folder(id=ASSETS_FOLDER_ID, output=str(ASSETS_DIR), quiet=False, use_cookies=False)
    print("Downloading original Colab clips folder...")
    gdown.download_folder(id=CLIPS_FOLDER_ID, output=str(INPUT_DIR), quiet=False, use_cookies=False)"""

new_asset_block = """if IS_GITHUB_ACTIONS:
    # The master workflow downloads assets/ before Part 3 starts.
    # Reuse those files to avoid a second Drive download and duplicate folder nesting.
    existing_assets = [path for path in ASSETS_DIR.rglob("*") if path.is_file()]
    if existing_assets:
        print(f"Using {len(existing_assets)} asset file(s) already downloaded by the workflow.")
    else:
        assets_url = os.environ.get("DRIVE_ASSETS_URL", "").strip()
        if not assets_url:
            raise RuntimeError(
                "No files were found in assets/ and DRIVE_ASSETS_URL is missing. "
                "Check the GitHub Actions secret and Drive folder sharing."
            )
        print("Downloading Part 3 assets from Google Drive into:", ASSETS_DIR)
        assets_download = gdown.download_folder(
            url=assets_url,
            output=str(ASSETS_DIR),
            quiet=False,
            use_cookies=False,
            remaining_ok=True,
        )
        existing_assets = [path for path in ASSETS_DIR.rglob("*") if path.is_file()]
        if not existing_assets:
            raise RuntimeError(
                "The Part 3 assets folder is still empty after the Drive download. "
                "Check DRIVE_ASSETS_URL and set the folder to Anyone with the link — Viewer."
            )
    print("Part 2 clips will be read from the workflow's clips/ folder; no second clip download is performed.")
else:
    ASSETS_FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"
    CLIPS_FOLDER_ID = "10DSjb9etdFzQTg_rftRJpJkC5jKBjsNT"
    print("Downloading original Colab assets folder...")
    gdown.download_folder(id=ASSETS_FOLDER_ID, output=str(ASSETS_DIR), quiet=False, use_cookies=False)
    print("Downloading original Colab clips folder...")
    gdown.download_folder(id=CLIPS_FOLDER_ID, output=str(INPUT_DIR), quiet=False, use_cookies=False)"""

patched_source = _replace_once(
    patched_source, old_asset_block, new_asset_block, "Part 3 Drive asset-download block"
)

# 2) Fix an accidental 9,800-pixel webcam width that can exhaust runner memory.
patched_source = _replace_once(
    patched_source,
    "WEBCAM_W, WEBCAM_H = 9800, 800",
    "WEBCAM_W, WEBCAM_H = 980, 800",
    "webcam dimensions",
)

# 3) Let the main video continue after a shorter webcam overlay ends.
patched_source = _replace_once(
    patched_source,
    "overlay=x=(W-w)/2:y=120:shortest=1[bg_with_webcam];",
    "overlay=x=(W-w)/2:y=120:eof_action=pass:shortest=0[bg_with_webcam];",
    "webcam overlay end-of-file handling",
)

# 4) Use FFmpeg's audio resampler/layout conversion instead of assuming every source
#    has both left and right channels. This supports mono and stereo inputs.
old_audio = "aresample=44100,pan=stereo|c0=c0|c1=c1,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo"
new_audio = "aresample=44100,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo"
patched_source = _replace_all_expected(
    patched_source, old_audio, new_audio, "mono/stereo audio conversion", 4
)

# 5) Make temporary names unique even when two categories contain the same filename.
patched_source = _replace_once(
    patched_source,
    "vid_id = abs(hash(vid_name))",
    "vid_id = int(hashlib.sha256(str(Path(INPUT).resolve()).encode('utf-8')).hexdigest()[:12], 16)",
    "per-video temporary identifier",
)

# 6) Make optional AI text generation safe: seek quickly, close the image, parse
#    fenced or plain JSON, cap line lengths, and always clean up the temporary frame.
old_ai_function = """def get_ai_text(video_path, vid_id):
    if not model or not GEMINI_API_KEY:
        return "AMAZING MOMENT", "WATCH UNTIL END"
    frame_path = f"temp_frame_{vid_id}.jpg"
    subprocess.run(["ffmpeg", "-y", "-i", video_path, "-ss", "00:00:02", "-vframes", "1", frame_path], capture_output=True)
    for attempt in range(2):
        try:
            time.sleep(2)
            img = Image.open(frame_path)
            prompt = "Analyze this image for a viral short. Give me 1 catchy top line (max 3 words) and 1 emotional bottom line (max 5 words). Return JSON: {'top': '...', 'sub': '...'}"
            response = model.generate_content([prompt, img])
            data = json.loads(response.text.replace("```json", "").replace("```", "").strip())
            return clean_for_ffmpeg(data['top']).upper(), clean_for_ffmpeg(data['sub']).upper()
        except:
            continue
    return "AMAZING MOMENT", "WATCH UNTIL END" """

# Match the source function without relying on a trailing space.
old_ai_function = old_ai_function.rstrip()
new_ai_function = """def get_ai_text(video_path, vid_id):
    fallback = ("AMAZING MOMENT", "WATCH UNTIL END")
    if not model or not GEMINI_API_KEY:
        return fallback

    frame_path = WORKSPACE / f".part3_frame_{vid_id}.jpg"
    prompt = (
        "Analyze this image for a viral short. Give 1 catchy top line (maximum 3 words) "
        "and 1 emotional bottom line (maximum 5 words). "
        "Return only JSON with keys top and sub."
    )
    try:
        frame_result = subprocess.run(
            [
                "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                "-ss", "2", "-i", video_path, "-frames:v", "1", str(frame_path)
            ],
            capture_output=True,
            text=True,
        )
        if frame_result.returncode != 0 or not frame_path.is_file():
            print(f"AI frame extraction unavailable for {video_path}; using fallback text.")
            return fallback

        for attempt in range(2):
            try:
                with Image.open(frame_path) as image:
                    response = model.generate_content([prompt, image.copy()])
                raw_text = (getattr(response, "text", "") or "").strip()
                cleaned = raw_text.replace("```json", "").replace("```", "").strip()
                if not cleaned.startswith("{"):
                    json_match = re.search(r"\\{.*\\}", cleaned, flags=re.DOTALL)
                    if json_match:
                        cleaned = json_match.group(0)
                data = json.loads(cleaned)
                top = clean_for_ffmpeg(str(data.get("top", ""))).upper().strip()
                sub = clean_for_ffmpeg(str(data.get("sub", ""))).upper().strip()
                top = " ".join(top.split()[:3])
                sub = " ".join(sub.split()[:5])
                if top and sub:
                    return top, sub
            except Exception as exc:
                print(f"AI text attempt {attempt + 1}/2 failed: {exc}")
        return fallback
    except Exception as exc:
        print(f"AI text generation failed; using fallback text: {exc}")
        return fallback
    finally:
        try:
            frame_path.unlink(missing_ok=True)
        except OSError:
            pass"""

patched_source = _replace_once(
    patched_source, old_ai_function, new_ai_function, "AI text-generation function"
)

# 7) Validate the configured font path; fall back to the font installed by the workflow.
old_font_block = """FONT = os.environ.get(
    "PART3_FONT_PATH",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
)"""
new_font_block = """FONT = os.environ.get("PART3_FONT_PATH", str(FONT_PATH))
if not Path(FONT).is_file():
    print(f"Configured font not found at {FONT}; falling back to {FONT_PATH}.")
    FONT = str(FONT_PATH)"""
patched_source = _replace_once(
    patched_source, old_font_block, new_font_block, "font-path validation"
)

# 8) Improve compatibility for players that need the MP4 index at the front of the file.
patched_source = _replace_once(
    patched_source,
    '"-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", FINAL_OUT',
    '"-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", "-movflags", "+faststart", FINAL_OUT',
    "final MP4 fast-start flag",
)

# 9) Verify every output file is non-empty and readable by ffprobe.
old_output_validation = """output_videos = sorted(
    str(path) for path in Path(OUTPUT_DIR).rglob("*")
    if path.is_file() and path.suffix.lower() == ".mp4"
)
print(f"Part 3 output MP4 count: {len(output_videos)}")
if not output_videos:
    raise RuntimeError(
        f"Part 3 processed {len(all_video_paths)} input video(s) but created no MP4 files in {OUTPUT_DIR}."
    )
print("\\n᎐᎐᎐ COMPLETED!")"""
new_output_validation = """output_videos = sorted(
    str(path) for path in Path(OUTPUT_DIR).rglob("*")
    if path.is_file() and path.suffix.lower() == ".mp4"
)
print(f"Part 3 output MP4 count: {len(output_videos)}")
if not output_videos:
    raise RuntimeError(
        f"Part 3 processed {len(all_video_paths)} input video(s) but created no MP4 files in {OUTPUT_DIR}."
    )

for output_path in output_videos:
    output_file = Path(output_path)
    if not output_file.is_file() or output_file.stat().st_size <= 0:
        raise RuntimeError(f"Part 3 created an empty or missing output file: {output_file}")
    probe = subprocess.run(
        [
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(output_file)
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(
            f"Part 3 output failed FFprobe validation: {output_file}\\n{probe.stderr[-1200:]}"
        )
    try:
        output_duration = float(probe.stdout.strip())
    except (TypeError, ValueError) as exc:
        raise RuntimeError(f"Could not read output duration for {output_file}: {probe.stdout!r}") from exc
    if output_duration <= 0:
        raise RuntimeError(f"Part 3 output has an invalid duration: {output_file}")

print(f"Part 3 output integrity check passed for {len(output_videos)} MP4 file(s).")
print("\\n᎐᎐᎐ COMPLETED!")"""
patched_source = _replace_once(
    patched_source, old_output_validation, new_output_validation, "final output validation"
)

# Compile before running any FFmpeg or network work.
compile(patched_source, "Part_3.ipynb:Part_3_v2_advanced", "exec")
print("All 9 compatibility/reliability patches applied and Python syntax check passed.")
print("Execution mode: sequential (one video at a time) to keep GitHub runner memory stable.")

exec(compile(patched_source, "Part_3.ipynb:Part_3_v2_advanced", "exec"), globals(), globals())
